# Randomness Validation: How Do We Know It's Truly Random?

**The statistical tests behind QuantumFair's Fairness Certificates**

---

Generating quantum random numbers is only half the problem. The other half: **how do we verify that the output is actually random?**

This notebook explains the theory behind randomness validation, implements the specific tests QuantumFair uses, and shows how they catch different types of non-randomness.

## 1. The Fundamental Problem

You cannot look at a finite sequence and declare it "truly random" — any finite sequence of bits has nonzero probability of appearing in both a random and a non-random source.

The sequence `0000000000` could come from a perfect quantum RNG. It's unlikely (probability 2⁻¹⁰ ≈ 0.1%), but possible.

What we *can* do is answer a different question:

> **"Is this sequence consistent with having been produced by a random process?"**

This is hypothesis testing. We define a **null hypothesis** (H₀: the source is random) and check whether the observed data is statistically consistent with H₀. If it's not, we reject and flag the sequence.

### What Tests Can and Cannot Do

| Can Do | Cannot Do |
|---|---|
| Detect bias (too many 0s or 1s) | Prove a source is random |
| Detect patterns (runs, repeats) | Distinguish good PRNG from QRNG |
| Detect correlations between bits | Guarantee future outputs are random |
| Flag hardware malfunctions | Replace Bell tests for quantum certification |

Statistical tests are **necessary but not sufficient**. They catch broken sources but can't prove true randomness. That's why QuantumFair uses them as a sanity check, while the Bell test (CHSH) provides the actual quantum proof.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(None)  # Use OS entropy for examples

def generate_good_random(n):
    """Simulate output from a working quantum RNG (fair coin)."""
    return np.random.randint(0, 2, size=n)

def generate_biased(n, p_one=0.55):
    """Simulate a biased source (hardware error)."""
    return (np.random.random(n) < p_one).astype(int)

def generate_correlated(n):
    """Simulate a source with sequential correlation (sticky bits)."""
    bits = [np.random.randint(0, 2)]
    for _ in range(n - 1):
        # 70% chance of repeating the previous bit
        if np.random.random() < 0.7:
            bits.append(bits[-1])
        else:
            bits.append(1 - bits[-1])
    return np.array(bits)

def generate_periodic(n, period=8):
    """Simulate a source with hidden periodicity."""
    pattern = np.random.randint(0, 2, size=period)
    return np.tile(pattern, n // period + 1)[:n]

# Generate test sequences
N = 10_000
seq_good = generate_good_random(N)
seq_biased = generate_biased(N, 0.55)
seq_correlated = generate_correlated(N)
seq_periodic = generate_periodic(N, period=16)

print("Test sequences generated:")
for name, seq in [('Good random', seq_good), ('Biased (55%)', seq_biased), 
                   ('Correlated', seq_correlated), ('Periodic (16)', seq_periodic)]:
    print(f"  {name:20s}: {''.join(map(str, seq[:50]))}...  P(1)={seq.mean():.3f}")

## 2. Test 1: Frequency (Monobit) Test

**What it checks:** Is the proportion of 0s and 1s close to 50/50?

**The theory:** For N truly random bits, the number of 1s follows a binomial distribution B(N, 0.5). By the Central Limit Theorem, for large N this is approximately normal:

$$\text{count}(1) \sim \mathcal{N}\left(\frac{N}{2},\; \frac{N}{4}\right)$$

We compute a z-score:

$$z = \frac{|\text{count}(1) - N/2|}{\sqrt{N/4}} = \frac{|S_n|}{\sqrt{N}}$$

where $S_n = \sum (2b_i - 1)$ converts bits to ±1 and sums them.

The p-value is $\text{erfc}(z / \sqrt{2})$. If p < 0.01, the sequence is too biased to be random.

**What it catches:** Biased sources, stuck qubits, systematic readout errors.

**What it misses:** Correlated bits that happen to have 50% ones, periodic sequences with equal counts.

In [ ]:
from scipy.special import erfc

def frequency_test(bits):
    """NIST SP 800-22 Frequency (Monobit) Test.
    
    H₀: The sequence is random (P(1) = P(0) = 0.5)
    
    Returns (p_value, z_score, pass/fail)
    """
    n = len(bits)
    # Convert 0→-1, 1→+1 and sum
    s_n = np.sum(2 * bits - 1)
    # Compute test statistic
    z = abs(s_n) / np.sqrt(n)
    # Two-sided p-value
    p_value = erfc(z / np.sqrt(2))
    return p_value, z, p_value >= 0.01


print(f"{'Sequence':<22} {'P(1)':>8} {'z-score':>10} {'p-value':>10} {'Result':>8}")
print("─" * 62)

for name, seq in [('Good random', seq_good), ('Biased (55%)', seq_biased),
                   ('Correlated', seq_correlated), ('Periodic (16)', seq_periodic)]:
    p, z, passed = frequency_test(seq)
    result = '✓ PASS' if passed else '✗ FAIL'
    color_result = result
    print(f"{name:<22} {seq.mean():>8.4f} {z:>10.4f} {p:>10.6f} {color_result:>8}")

print(f"\nThreshold: p-value ≥ 0.01 → PASS")
print(f"Note: Correlated sequences can PASS this test if they happen to have ~50% ones.")
print(f"That's why we need additional tests.")

In [ ]:
# Visualize: distribution of z-scores for truly random sequences
# vs. where biased/broken sources land

n_trials = 2000
z_random = [frequency_test(generate_good_random(N))[1] for _ in range(n_trials)]
z_biased = [frequency_test(generate_biased(N, 0.53))[1] for _ in range(n_trials)]

fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(z_random, bins=50, alpha=0.7, color='#4CAF50', label='Random source (should pass)', 
        density=True, edgecolor='black')
ax.hist(z_biased, bins=50, alpha=0.7, color='#F44336', label='Biased source (53% ones)', 
        density=True, edgecolor='black')

# Critical value for p=0.01 (two-sided)
z_crit = 2.576
ax.axvline(x=z_crit, color='red', linewidth=2, linestyle='--', label=f'Rejection threshold (z={z_crit})')
ax.fill_betweenx([0, 1], z_crit, ax.get_xlim()[1] if ax.get_xlim()[1] > z_crit else 8, 
                 alpha=0.15, color='red', label='Rejection region (p < 0.01)')

ax.set_xlabel('z-score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Frequency Test: Random vs Biased Sources\n(z-score distribution over 2000 trials)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('frequency_test.png', dpi=100, bbox_inches='tight')
plt.show()

pct_random_pass = sum(1 for z in z_random if z < z_crit) / len(z_random) * 100
pct_biased_pass = sum(1 for z in z_biased if z < z_crit) / len(z_biased) * 100
print(f"Random source pass rate:  {pct_random_pass:.1f}% (expected ~99%)")
print(f"Biased source pass rate:  {pct_biased_pass:.1f}% (should be much lower)")

## 3. Test 2: Runs Test

**What it checks:** Are there too many or too few "runs" (consecutive sequences of identical bits)?

**The theory:** A "run" is a maximal sequence of consecutive identical bits. For example:

```
Sequence: 1 1 0 0 0 1 0 1 1 1
Runs:     [11][000][1][0][111]
Run count: 5
```

For a truly random binary sequence of length N with proportion π = count(1)/N:

$$E[\text{runs}] = 2N\pi(1-\pi) + 1$$

$$\text{Var}[\text{runs}] = \frac{2N\pi(1-\pi)(2N\pi(1-\pi) - 1)}{N-1} \approx 2N\pi(1-\pi)$$

The z-score is:

$$z = \frac{|R - E[\text{runs}]|}{\sqrt{\text{Var}[\text{runs}]}}$$

Too few runs → bits are "sticky" (correlated, tend to repeat).
Too many runs → bits are "alternating" (anti-correlated).

**What it catches:** Sequential correlation, stuck-high/stuck-low qubits, T1 decay artifacts.

**What it misses:** Bias without correlation, periodicity that happens to have normal run counts.

In [ ]:
def runs_test(bits):
    """NIST SP 800-22 Runs Test.
    
    Prerequisite: Frequency test should pass first (|π - 0.5| < 2/√N).
    
    H₀: The sequence has no sequential correlation.
    
    Returns (p_value, observed_runs, expected_runs, pass/fail)
    """
    n = len(bits)
    pi = np.mean(bits)
    
    # Prerequisite check
    tau = 2.0 / np.sqrt(n)
    if abs(pi - 0.5) >= tau:
        return 0.0, 0, 0, False  # Fails prerequisite (too biased for runs test)
    
    # Count runs
    runs = 1
    for i in range(1, n):
        if bits[i] != bits[i-1]:
            runs += 1
    
    # Expected runs and variance
    expected = 2 * n * pi * (1 - pi) + 1
    variance = 2 * n * pi * (1 - pi) * (2 * n * pi * (1 - pi) - 1) / (n - 1)
    
    # z-score and p-value
    z = abs(runs - expected) / np.sqrt(variance)
    p_value = erfc(z / np.sqrt(2))
    
    return p_value, runs, expected, p_value >= 0.01


print(f"{'Sequence':<22} {'Runs':>8} {'Expected':>10} {'p-value':>10} {'Result':>8}")
print("─" * 62)

for name, seq in [('Good random', seq_good), ('Biased (55%)', seq_biased),
                   ('Correlated', seq_correlated), ('Periodic (16)', seq_periodic)]:
    p, runs, expected, passed = runs_test(seq)
    result = '✓ PASS' if passed else '✗ FAIL'
    print(f"{name:<22} {runs:>8} {expected:>10.1f} {p:>10.6f} {result:>8}")

print(f"\nThe runs test catches the CORRELATED sequence that the frequency test missed.")
print(f"Correlated bits produce too FEW runs (bits 'stick' together).")

In [ ]:
# Visualize: what runs look like for random vs correlated sequences

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)

display_n = 200
sequences = [
    ('Good random', seq_good[:display_n], '#4CAF50'),
    ('Biased (55%)', seq_biased[:display_n], '#FF9800'),
    ('Correlated (sticky)', seq_correlated[:display_n], '#F44336'),
    ('Periodic (16-bit)', seq_periodic[:display_n], '#9C27B0'),
]

for ax, (name, seq, color) in zip(axes, sequences):
    ax.step(range(len(seq)), seq, where='mid', color=color, linewidth=0.8)
    ax.fill_between(range(len(seq)), seq, step='mid', alpha=0.3, color=color)
    ax.set_ylabel(name, fontsize=9, rotation=0, ha='right', va='center')
    ax.set_ylim(-0.2, 1.2)
    ax.set_yticks([0, 1])
    ax.grid(True, alpha=0.2)
    
    # Count runs in displayed portion
    runs = 1 + sum(1 for i in range(1, len(seq)) if seq[i] != seq[i-1])
    ax.text(display_n + 2, 0.5, f'{runs} runs', fontsize=10, va='center',
           fontweight='bold', color=color)

axes[-1].set_xlabel('Bit position', fontsize=11)
plt.suptitle('Bit Sequences: Visual Comparison\n(First 200 bits of each sequence)', 
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('runs_visual.png', dpi=100, bbox_inches='tight')
plt.show()

print("Random sequences have many short runs with occasional longer ones.")
print("Correlated sequences have long runs (bits stick together).")
print("Periodic sequences have a visually obvious repeating pattern.")

## 4. Test 3: Chi-Squared Test on Block Distribution

**What it checks:** When we group bits into K-bit blocks, is the distribution of block values uniform?

**The theory:** For a truly random sequence, every K-bit block should appear with equal probability 1/2ᴷ. We count the occurrences of each possible block value and compare to the expected uniform distribution using the chi-squared statistic:

$$\chi^2 = \sum_{i=0}^{2^K - 1} \frac{(O_i - E_i)^2}{E_i}$$

where $O_i$ is the observed count of block value $i$ and $E_i = N / 2^K$ is the expected count.

Under H₀, $\chi^2 \sim \chi^2_{2^K - 1}$ (chi-squared distribution with $2^K - 1$ degrees of freedom).

**Block size K in QuantumFair:** We use K=4 (16 possible blocks). This balances sensitivity (detects patterns at the nibble level) with statistical power (N/16 expected count per block is large enough for reliable chi-squared).

**What it catches:** Non-uniform distribution at the block level, subtle patterns that frequency and runs tests miss, periodic sequences.

**What it misses:** Correlations between blocks (sequential patterns spanning multiple blocks).

In [ ]:
def chi_squared_test(bits, block_size=4):
    """Chi-squared test for uniformity of K-bit blocks.
    
    Groups bits into non-overlapping blocks of size K.
    Tests whether block values are uniformly distributed.
    
    H₀: All 2^K block values are equally likely.
    
    Returns (p_value, chi2_stat, degrees_of_freedom, pass/fail)
    """
    n = len(bits)
    num_blocks = n // block_size
    num_categories = 2 ** block_size
    
    # Convert bit blocks to integers
    block_values = []
    for i in range(num_blocks):
        block = bits[i * block_size : (i + 1) * block_size]
        value = sum(b * (2 ** (block_size - 1 - j)) for j, b in enumerate(block))
        block_values.append(value)
    
    # Count occurrences
    counts = Counter(block_values)
    observed = [counts.get(i, 0) for i in range(num_categories)]
    expected = num_blocks / num_categories
    
    # Chi-squared statistic
    chi2 = sum((o - expected) ** 2 / expected for o in observed)
    
    # Degrees of freedom = 2^K - 1
    dof = num_categories - 1
    
    # p-value from chi-squared CDF
    p_value = 1 - stats.chi2.cdf(chi2, dof)
    
    return p_value, chi2, dof, p_value >= 0.01, observed, expected


print(f"{'Sequence':<22} {'χ²':>10} {'dof':>6} {'p-value':>10} {'Result':>8}")
print("─" * 60)

for name, seq in [('Good random', seq_good), ('Biased (55%)', seq_biased),
                   ('Correlated', seq_correlated), ('Periodic (16)', seq_periodic)]:
    p, chi2, dof, passed, obs, exp = chi_squared_test(seq, block_size=4)
    result = '✓ PASS' if passed else '✗ FAIL'
    print(f"{name:<22} {chi2:>10.2f} {dof:>6} {p:>10.6f} {result:>8}")

print(f"\nBlock size: 4 bits → 16 possible block values")
print(f"Each block should appear ~{N // 16} times in {N} bits")
print(f"\nThe chi-squared test catches the PERIODIC sequence that has")
print(f"a severely non-uniform block distribution.")

In [ ]:
# Visualize: block distributions for each sequence type

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sequences = [
    ('Good Random', seq_good, '#4CAF50'),
    ('Biased (55%)', seq_biased, '#FF9800'),
    ('Correlated', seq_correlated, '#F44336'),
    ('Periodic (16-bit)', seq_periodic, '#9C27B0'),
]

for ax, (name, seq, color) in zip(axes.flat, sequences):
    p, chi2, dof, passed, observed, expected = chi_squared_test(seq, block_size=4)
    
    x = range(16)
    labels = [f'{i:04b}' for i in range(16)]
    
    bars = ax.bar(x, observed, color=color, alpha=0.7, edgecolor='black', label='Observed')
    ax.axhline(y=expected, color='black', linestyle='--', linewidth=2, 
              label=f'Expected ({expected:.0f})')
    
    # Highlight bars that deviate significantly
    for i, (bar, obs_val) in enumerate(zip(bars, observed)):
        deviation = abs(obs_val - expected) / np.sqrt(expected)
        if deviation > 2:  # More than 2σ
            bar.set_edgecolor('red')
            bar.set_linewidth(2)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, fontsize=7)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_xlabel('4-bit block value', fontsize=10)
    result = '✓ PASS' if passed else '✗ FAIL'
    ax.set_title(f'{name}\nχ²={chi2:.1f}, p={p:.4f} → {result}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Chi-Squared Test: 4-bit Block Distributions\n(Each block value should appear equally often)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chi_squared_blocks.png', dpi=100, bbox_inches='tight')
plt.show()

print("The periodic sequence has a catastrophically non-uniform distribution — ")
print("only a few block values ever appear, because the pattern repeats every 16 bits.")

## 5. Test 4: CHSH Bell Test (Quantum Origin Proof)

This is the test that separates QuantumFair from every other randomness source. The CHSH test doesn't check whether the output *looks* random — it checks whether the output *could have come from a classical source at all*.

### The CHSH Inequality

Consider two entangled qubits (a Bell pair). Alice measures qubit A with one of two measurement settings (a₁ or a₂). Bob measures qubit B with one of two settings (b₁ or b₂). Each measurement produces +1 or -1.

Define the **correlation** for settings (a, b):

$$E(a, b) = \langle A \cdot B \rangle = P(\text{same}) - P(\text{different})$$

The **CHSH parameter**:

$$S = |E(a_1, b_1) - E(a_1, b_2) + E(a_2, b_1) + E(a_2, b_2)|$$

**Bell's theorem proves:** For ANY classical local hidden variable theory:

$$S \leq 2$$

**Quantum mechanics predicts:** With optimal measurement angles:

$$S = 2\sqrt{2} \approx 2.828$$

If we measure S > 2 from our quantum hardware, it constitutes a mathematical proof that no classical system produced the correlations. The randomness is certified quantum.

### Measurement Angles

The optimal CHSH angles (maximizing violation):

| Setting | Angle | Purpose |
|---|---|---|
| a₁ (Alice) | 0° | Reference |
| a₂ (Alice) | 45° (π/4) | Rotated |
| b₁ (Bob) | 22.5° (π/8) | Between Alice's settings |
| b₂ (Bob) | 67.5° (3π/8) | Between Alice's settings |

In [ ]:
def simulate_bell_test(n_shots_per_setting=2500, noise_level=0.05):
    """
    Simulate a CHSH Bell test as performed on IBM quantum hardware.
    
    Simulates the quantum mechanical predictions with realistic noise
    to show what QuantumFair would measure on real hardware.
    """
    # Optimal CHSH measurement angles
    settings = [
        (0, np.pi/8),           # (a1, b1)
        (0, 3*np.pi/8),         # (a1, b2)
        (np.pi/4, np.pi/8),     # (a2, b1)
        (np.pi/4, 3*np.pi/8),   # (a2, b2)
    ]
    
    correlations = []
    raw_results = {}
    
    for i, (a, b) in enumerate(settings):
        # Quantum prediction for |Φ+⟩ state: E(a,b) = -cos(2(a-b))
        E_ideal = -np.cos(2 * (a - b))
        
        # Simulate individual shots with noise
        # P(same outcome) = (1 - E) / 2, P(different) = (1 + E) / 2
        # (for anti-correlated Bell state)
        p_same = (1 + E_ideal) / 2
        
        # Add hardware noise (depolarizing-like)
        p_same_noisy = p_same * (1 - noise_level) + 0.5 * noise_level
        
        # Generate shot outcomes
        outcomes = np.random.random(n_shots_per_setting) < p_same_noisy
        same_count = outcomes.sum()
        diff_count = n_shots_per_setting - same_count
        
        E_measured = (same_count - diff_count) / n_shots_per_setting
        correlations.append(E_measured)
        
        raw_results[f'setting_{i}'] = {
            'angles': (a, b),
            'E_ideal': E_ideal,
            'E_measured': E_measured,
            'same': int(same_count),
            'different': int(diff_count),
        }
    
    # CHSH S parameter
    E11, E12, E21, E22 = correlations
    S = abs(E11 - E12 + E21 + E22)
    
    # Statistical significance: standard error of S
    # Each E has stderr ≈ 1/√n, S combines 4 independent E values
    se_S = 2.0 / np.sqrt(n_shots_per_setting)  # approximate
    sigma_violation = (S - 2.0) / se_S
    
    return S, correlations, raw_results, se_S, sigma_violation


# Run a Bell test
S, corrs, results, se, sigma = simulate_bell_test(n_shots_per_setting=5000, noise_level=0.04)

print("CHSH Bell Test Results")
print("=" * 65)
print(f"{'Setting':<12} {'Angles (a,b)':<20} {'E_ideal':>10} {'E_measured':>12}")
print("─" * 65)
for key, r in results.items():
    a_deg = np.degrees(r['angles'][0])
    b_deg = np.degrees(r['angles'][1])
    print(f"{key:<12} ({a_deg:>5.1f}°, {b_deg:>5.1f}°){' ':>5} {r['E_ideal']:>10.4f} {r['E_measured']:>12.4f}")

print("─" * 65)
print(f"\nCHSH S = {S:.4f}")
print(f"Classical limit: S ≤ 2.000")
print(f"Quantum maximum: S = 2√2 ≈ 2.828")
print(f"Violation: {S - 2:.4f} above classical limit")
print(f"Significance: {sigma:.1f}σ (standard errors above classical limit)")
print(f"\nVerdict: {'✓ QUANTUM CERTIFIED' if S > 2 else '✗ NOT CERTIFIED'}")

In [ ]:
# Visualize the CHSH test results as QuantumFair would display them

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Gauge showing S value
theta = np.linspace(0, np.pi, 100)
ax1.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2)

# Scale: S ranges from 0 to 2√2 ≈ 2.83, map to 0 to π
s_max = 2 * np.sqrt(2)

# Classical region (0 to 2)
theta_classical = np.linspace(0, np.pi * (2 / s_max), 50)
ax1.fill_between(np.cos(theta_classical), 0, np.sin(theta_classical), 
                alpha=0.2, color='#2196F3', label='Classical region (S ≤ 2)')

# Quantum region (2 to 2√2)
theta_quantum = np.linspace(np.pi * (2 / s_max), np.pi, 50)
ax1.fill_between(np.cos(theta_quantum), 0, np.sin(theta_quantum), 
                alpha=0.3, color='#FF5722', label='Quantum-only region (S > 2)')

# Classical limit line
angle_limit = np.pi * (2 / s_max)
ax1.plot([0, np.cos(angle_limit)], [0, np.sin(angle_limit)], 
        'r--', linewidth=2, label='Bell limit (S = 2)')

# Needle for measured S
angle_measured = np.pi * (S / s_max)
ax1.annotate('', xy=(0.85*np.cos(angle_measured), 0.85*np.sin(angle_measured)), 
            xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='black', lw=3))

ax1.text(np.cos(angle_measured)*1.05, np.sin(angle_measured)*1.05, 
        f'S = {S:.3f}', fontsize=14, fontweight='bold', ha='center')

# Labels
ax1.text(0.5, -0.15, 'Classical', fontsize=11, ha='center', color='#2196F3', fontweight='bold')
ax1.text(-0.7, 0.3, 'Quantum', fontsize=11, ha='center', color='#FF5722', fontweight='bold')

ax1.set_xlim(-1.3, 1.3)
ax1.set_ylim(-0.3, 1.3)
ax1.set_aspect('equal')
ax1.set_title('CHSH Gauge\n(as rendered in QuantumFair LWC)', fontsize=13, fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.axis('off')

# Right: Correlation values for each setting pair
setting_labels = ['E(a₁,b₁)', 'E(a₁,b₂)', 'E(a₂,b₁)', 'E(a₂,b₂)']
ideal_E = [-np.cos(2*(a-b)) for a, b in [(0, np.pi/8), (0, 3*np.pi/8), 
                                           (np.pi/4, np.pi/8), (np.pi/4, 3*np.pi/8)]]

x_pos = np.arange(4)
width = 0.35

ax2.bar(x_pos - width/2, ideal_E, width, color='#42A5F5', label='Quantum prediction', 
       edgecolor='black', alpha=0.8)
ax2.bar(x_pos + width/2, corrs, width, color='#FF7043', label='Measured', 
       edgecolor='black', alpha=0.8)

ax2.set_xticks(x_pos)
ax2.set_xticklabels(setting_labels, fontsize=11)
ax2.set_ylabel('Correlation E(a,b)', fontsize=12)
ax2.set_title('Measurement Correlations by Setting\n(Ideal vs Measured with hardware noise)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(-1, 1)
ax2.axhline(y=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig('chsh_results.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Putting It All Together: QuantumFair's Validation Pipeline

When QuantumFair receives measurement results from IBM Quantum hardware, it runs the following validation pipeline before issuing a Fairness Certificate:

```
Raw IBM Measurement Results
         │
         ▼
┌─────────────────────────┐
│  Von Neumann Debiasing  │ ← Removes hardware bias
│  (01→0, 10→1, discard)  │    Output: perfectly unbiased bits
└────────────┬────────────┘
             │
             ▼
┌─────────────────────────┐
│  Test 1: Frequency      │ ← Is P(1) ≈ 0.5?
│  (monobit z-test)       │    Catches: bias, stuck qubits
└────────────┬────────────┘
             │ PASS
             ▼
┌─────────────────────────┐
│  Test 2: Runs           │ ← Are consecutive bits independent?
│  (sequential correlation)│   Catches: sticky bits, T1 decay
└────────────┬────────────┘
             │ PASS
             ▼
┌─────────────────────────┐
│  Test 3: Chi-Squared    │ ← Are 4-bit blocks uniformly distributed?
│  (block distribution)   │    Catches: periodicity, patterns
└────────────┬────────────┘
             │ PASS
             ▼
┌─────────────────────────┐
│  Test 4: CHSH Bell Test │ ← Is S > 2.0?
│  (quantum origin proof) │    Catches: classical substitution
│  [Bell-test mode only]  │    Proves: quantum mechanical origin
└────────────┬────────────┘
             │ PASS
             ▼
┌─────────────────────────┐
│  ✓ FAIRNESS CERTIFICATE │
│  ISSUED                 │
│                         │
│  Includes:              │
│  • All test p-values    │
│  • CHSH S-value         │
│  • SHA-256 hashes       │
│  • IBM Job ID           │
└─────────────────────────┘
```

If **any** statistical test fails (p < 0.01), the certificate is **not issued** and the user is prompted to retry. This typically indicates a temporary hardware issue (calibration drift, elevated error rates).

### Why These Four Tests?

| Test | Detects | Doesn't Detect | NIST SP 800-22 |
|---|---|---|---|
| Frequency | Global bias | Patterns, correlation | Test 1 |
| Runs | Sequential correlation | Global bias, periodicity | Test 2 |
| Chi-squared | Block-level non-uniformity | Inter-block correlation | Test 3 (variant) |
| CHSH Bell test | Classical source substitution | Nothing (conclusive proof) | N/A (physics, not statistics) |

Each test covers a blind spot of the others. Together, they form a comprehensive validation that the randomness is both **statistically valid** and **physically quantum**.

In [ ]:
def full_validation_pipeline(bits, bell_test=True, verbose=True):
    """Run the complete QuantumFair validation pipeline.
    
    This is a simulation of what QRNGValidator.cls does in Apex.
    """
    results = {}
    all_passed = True
    
    if verbose:
        print("QuantumFair Validation Pipeline")
        print("=" * 55)
        print(f"Input: {len(bits):,} bits")
        print()
    
    # Test 1: Frequency
    p, z, passed = frequency_test(bits)
    results['frequency'] = {'p_value': p, 'z_score': z, 'passed': passed}
    if verbose:
        status = '✓ PASS' if passed else '✗ FAIL'
        print(f"Test 1 — Frequency:    p={p:.6f}  z={z:.4f}  {status}")
    if not passed:
        all_passed = False
    
    # Test 2: Runs
    p, runs, expected, passed = runs_test(bits)
    results['runs'] = {'p_value': p, 'runs': runs, 'expected': expected, 'passed': passed}
    if verbose:
        status = '✓ PASS' if passed else '✗ FAIL'
        print(f"Test 2 — Runs:         p={p:.6f}  runs={runs}  {status}")
    if not passed:
        all_passed = False
    
    # Test 3: Chi-squared
    p, chi2, dof, passed, obs, exp = chi_squared_test(bits, block_size=4)
    results['chi_squared'] = {'p_value': p, 'chi2': chi2, 'dof': dof, 'passed': passed}
    if verbose:
        status = '✓ PASS' if passed else '✗ FAIL'
        print(f"Test 3 — Chi-squared:  p={p:.6f}  χ²={chi2:.2f}  {status}")
    if not passed:
        all_passed = False
    
    # Test 4: CHSH (simulated)
    if bell_test:
        S, corrs, raw, se, sigma = simulate_bell_test(n_shots_per_setting=2500, noise_level=0.04)
        bell_passed = S > 2.0
        results['chsh'] = {'S': S, 'sigma': sigma, 'passed': bell_passed}
        if verbose:
            status = '✓ PASS' if bell_passed else '✗ FAIL'
            print(f"Test 4 — CHSH Bell:    S={S:.4f}  ({sigma:.1f}σ)  {status}")
        if not bell_passed:
            all_passed = False
    
    if verbose:
        print()
        print("─" * 55)
        if all_passed:
            cert_type = 'QUANTUM CERTIFIED' if bell_test else 'STATISTICALLY VALIDATED'
            print(f"Result: ✓ {cert_type}")
            print(f"Fairness Certificate: ISSUED")
        else:
            print(f"Result: ✗ VALIDATION FAILED")
            print(f"Fairness Certificate: NOT ISSUED")
            print(f"Recommendation: Retry (possible hardware calibration issue)")
    
    return all_passed, results


# Run the pipeline on good random data
good_bits = generate_good_random(10_000)
full_validation_pipeline(good_bits, bell_test=True)

In [ ]:
# Now run it on each type of bad data to show how validation catches problems

print("\n" + "=" * 55)
print("TESTING AGAINST KNOWN BAD SOURCES")
print("=" * 55)

bad_sources = [
    ("Biased source (55%)", generate_biased(10_000, 0.55)),
    ("Correlated source", generate_correlated(10_000)),
    ("Periodic source (16)", generate_periodic(10_000, 16)),
    ("All zeros (stuck)", np.zeros(10_000, dtype=int)),
    ("Alternating 0101...", np.tile([0, 1], 5_000)),
]

for name, bits in bad_sources:
    print(f"\n{'─' * 55}")
    print(f"Source: {name}")
    print(f"{'─' * 55}")
    full_validation_pipeline(bits, bell_test=False)

In [ ]:
# Final visualization: success rates across many trials for each source type

n_trials = 500
n_bits = 5_000

sources = {
    'Good\nrandom': lambda: generate_good_random(n_bits),
    'Biased\n(52%)': lambda: generate_biased(n_bits, 0.52),
    'Biased\n(55%)': lambda: generate_biased(n_bits, 0.55),
    'Correlated\n(70% sticky)': lambda: generate_correlated(n_bits),
    'Periodic\n(16-bit)': lambda: generate_periodic(n_bits, 16),
}

test_names = ['Frequency', 'Runs', 'Chi-squared']
pass_rates = {src: {t: 0 for t in test_names} for src in sources}
overall_rates = {src: 0 for src in sources}

for src_name, generator in sources.items():
    for _ in range(n_trials):
        bits = generator()
        
        p1, _, passed1 = frequency_test(bits)
        p2, _, _, passed2 = runs_test(bits)
        p3, _, _, passed3, _, _ = chi_squared_test(bits)
        
        if passed1: pass_rates[src_name]['Frequency'] += 1
        if passed2: pass_rates[src_name]['Runs'] += 1
        if passed3: pass_rates[src_name]['Chi-squared'] += 1
        if passed1 and passed2 and passed3: overall_rates[src_name] += 1

# Convert to percentages
for src in sources:
    for t in test_names:
        pass_rates[src][t] = pass_rates[src][t] / n_trials * 100
    overall_rates[src] = overall_rates[src] / n_trials * 100

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(sources))
width = 0.2

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
all_test_names = test_names + ['All tests']

for i, test in enumerate(test_names):
    values = [pass_rates[src][test] for src in sources]
    ax.bar(x + i * width, values, width, label=test, color=colors[i], 
          edgecolor='black', alpha=0.85)

# Overall pass rate
overall_values = [overall_rates[src] for src in sources]
ax.bar(x + 3 * width, overall_values, width, label='All tests combined', 
      color=colors[3], edgecolor='black', alpha=0.85)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(sources.keys(), fontsize=10)
ax.set_ylabel('Pass rate (%)', fontsize=12)
ax.set_title(f'Validation Test Pass Rates by Source Type\n({n_trials} trials × {n_bits:,} bits each)', 
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
ax.axhline(y=99, color='green', linestyle=':', alpha=0.5, linewidth=1)
ax.text(len(sources)-0.5, 101, 'Expected pass rate (~99%)', fontsize=8, color='green')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('validation_pass_rates.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key observations:")
print(f"• Good random passes all tests ~{overall_rates[list(sources.keys())[0]]:.0f}% of the time (expected: ~99%)")
print(f"• Even subtle 52% bias is caught by the frequency test")
print(f"• Correlated sources pass frequency but fail runs")
print(f"• Periodic sources fail chi-squared catastrophically")
print(f"• The combined test suite catches ALL tested failure modes")

## 7. Why These Tests Are Sufficient for QuantumFair

NIST SP 800-22 defines 15 statistical tests for randomness. QuantumFair uses only 3 (plus the Bell test). Why?

### Scope matters

NIST SP 800-22 was designed to evaluate **general-purpose random number generators** — PRNGs, hardware RNGs, and other black-box sources. It needs 15 tests because it must catch every conceivable class of non-randomness from an unknown source.

QuantumFair's source is **not unknown**. It is:
- A specific, known quantum circuit (Hadamard + measurement)
- Running on specific, characterized hardware (IBM Heron r2)
- With known error modes (readout error, gate error, T1 decay, crosstalk)

The failure modes of this specific source are:

| Failure Mode | Caught By |
|---|---|
| Readout bias (most common) | Frequency test + von Neumann debiasing |
| Qubit decay during measurement | Runs test (creates correlation) |
| Crosstalk between qubits | Chi-squared test (creates block patterns) |
| Hardware malfunction | Any test will fail |
| Classical substitution attack | CHSH Bell test (impossible to fake) |

Our three statistical tests cover the known error modes of the specific hardware. The Bell test provides the unforgeable quantum proof. Additional NIST tests (longest run, spectral, etc.) would add marginal detection power for failure modes that don't occur on this hardware.

### The Bell test is the real validation

Statistical tests tell us: *"This sequence is consistent with randomness."*

The Bell test tells us: *"This randomness was produced by quantum mechanics."*

No amount of statistical testing can distinguish a perfect PRNG from true quantum randomness. Only the Bell test can. That's why it's the cornerstone of QuantumFair's certification — the statistical tests are sanity checks; the Bell test is the proof.